# 04 · Negativos y reglas de armonía (fase 2: TTMn y DLMn)

Etapa 4. Prepara las dos etiquetas de la fase 2 (revisión §12.4):
- **TTMn (real o falso):** 1 = outfit real de Polyvore; 0 = negativo armado con prendas del **mismo slot** y el **mismo número de prendas** (`src/modelo_fly/negativos.py`).
- **DLMn (armonía 0–7):** número de las 7 reglas que el outfit **no viola**; una regla que no aplica cuenta como cumplida (`src/modelo_fly/reglas.py`).

**Negativos (2 por outfit real, siempre con productos de la misma partición del paso 4b):**
- `aleatorio`: se cambian **todas** las prendas por otras del mismo slot (protocolo de Han et al., 2017).
- `dificil`: se cambia **una** prenda por otra del mismo slot y, si se conoce, del mismo estilo.

**Entradas:** `polyvore_items_atributos.csv` (01) y `particion_outfits.csv` (03).
**Salidas** (se suben a git, para que las etapas siguientes no necesiten descargar Polyvore):
`data/processed/hito3/fase2_productos.csv` (un producto por fila con sus atributos) y `data/processed/hito3/fase2_outfits.csv` (reales + negativos, con reglas y score).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from modelo_fly.negativos import TIPOS, generar
from modelo_fly.reglas import DESCRIPCION, REGLAS, evaluar

PROC = ROOT / "data" / "processed" / "hito3"
ATRIBUTOS = ["slot", "tono", "familia_color", "patron", "material", "estilo", "calce", "epoca", "largo"]

items = pd.read_csv(PROC / "polyvore_items_atributos.csv", dtype=str, keep_default_na=False)
particion = pd.read_csv(PROC / "particion_outfits.csv", dtype=str)
items = items.merge(particion[["set_id", "particion"]], on="set_id", validate="many_to_one")
items["index"] = items["index"].astype(int)

# Un mismo producto tiene siempre los mismos atributos (se verifica), así que se trabaja por producto.
assert items.groupby("tid")[ATRIBUTOS].nunique().max().max() == 1
productos = items.drop_duplicates("tid")[["tid", "particion", "categoria", "name", *ATRIBUTOS]].reset_index(drop=True)
reales = (items.sort_values(["set_id", "index"]).groupby("set_id")
          .agg(particion=("particion", "first"), tids=("tid", list)).reset_index())
print("Outfits reales:", len(reales), "| productos:", len(productos))
display(productos["particion"].value_counts().to_frame("productos"))

Outfits reales: 15012 | productos: 65366


,productos
particion,
train,43534
test,11033
valid,10799


## 1. Negativos

In [2]:
negativos = generar(reales, productos, semilla=2026)
print(negativos["tipo"].value_counts().to_dict())

# Controles: mismos slots que el real, misma partición, ningún conjunto igual a uno real, sin repetidos.
slot_de = dict(zip(productos["tid"], productos["slot"]))
part_de = dict(zip(productos["tid"], productos["particion"]))
tids_real = dict(zip(reales["set_id"], reales["tids"]))
mismos_slots = [sorted(slot_de[t] for t in n) == sorted(slot_de[t] for t in tids_real[o])
                for n, o in zip(negativos["tids"], negativos["set_id_origen"])]
misma_part = [all(part_de[t] == p for t in n) for n, p in zip(negativos["tids"], negativos["particion"])]
claves_reales = {frozenset(t) for t in reales["tids"]}
print("Mismos slots que su real:", f"{np.mean(mismos_slots):.1%}")
print("Todos los productos en su partición:", f"{np.mean(misma_part):.1%}")
print("Negativos iguales a un outfit real:", sum(frozenset(t) in claves_reales for t in negativos["tids"]))
print("Negativos repetidos:", len(negativos) - len({frozenset(t) for t in negativos["tids"]}))
print("Faltan (no se pudo armar):", len(reales) * len(TIPOS) - len(negativos))
assert all(mismos_slots) and all(misma_part)

{'aleatorio': 15012, 'dificil': 15012}


Mismos slots que su real: 100.0%
Todos los productos en su partición: 100.0%
Negativos iguales a un outfit real: 0
Negativos repetidos: 0
Faltan (no se pudo armar): 0


## 2. Reglas de armonía

Se evalúan sobre las prendas con el atributo conocido. P1 y P2 son **aproximaciones**: Polyvore no trae la escala del estampado ni su color de fondo (se usa el tono y el tipo de patrón).

In [3]:
display(pd.DataFrame({"regla": REGLAS, "criterio": [DESCRIPCION[r] for r in REGLAS]}).set_index("regla"))

reales_t = reales.assign(outfit_id=reales["set_id"], tipo="real", set_id_origen=reales["set_id"], cambiadas=0)
outfits = pd.concat([reales_t, negativos], ignore_index=True)
outfits["real"] = (outfits["tipo"] == "real").astype(int)
outfits["n_items"] = outfits["tids"].str.len()
atrib = productos.set_index("tid")[ATRIBUTOS].to_dict("index")
reglas = pd.DataFrame([evaluar([atrib[t] for t in tids]) for tids in outfits["tids"]])
outfits = pd.concat([outfits, reglas], axis=1)

orden = ["real", "dificil", "aleatorio"]
display(outfits.groupby("tipo")["score"].describe().reindex(orden).round(3))
display(pd.crosstab(outfits["tipo"], outfits["score"], normalize="index").mul(100).round(1).reindex(orden))

,criterio
regla,
C1,Máximo dos colores no neutros y un solo acento...
C2,"Si hay dos o más colores no neutros, al menos ..."
P1,Dos estampados distintos solo si comparten un ...
P2,No repetir el mismo tipo de estampado
P3,"Si hay dos o más prendas estampadas, al menos ..."
E1,Un solo elemento con volumen (oversize)
Ep1,No replicar una sola década de pies a cabeza


,count,mean,std,min,25%,50%,75%,max
tipo,,,,,,,,
real,15012.0,6.768,0.517,3.0,7.0,7.0,7.0,7.0
dificil,15012.0,6.740,0.541,3.0,7.0,7.0,7.0,7.0
aleatorio,15012.0,6.678,0.578,3.0,6.0,7.0,7.0,7.0


score,3,4,5,6,7
tipo,,,,,
real,0.0,0.4,3.1,15.7,80.8
dificil,0.0,0.4,3.4,17.7,78.4
aleatorio,0.1,0.5,3.7,22.9,72.8


In [4]:
filas = []
for r in REGLAS:
    fila = {"regla": r}
    for t in orden:
        d = outfits[outfits["tipo"] == t]
        fila[f"aplica_{t}_%"] = round(100 * d[f"aplica_{r}"].mean(), 1)
        fila[f"viola_{t}_%"] = round(100 * (1 - d[f"cumple_{r}"]).mean(), 1)
    filas.append(fila)
por_regla = pd.DataFrame(filas).set_index("regla")
display(por_regla)

,aplica_real_%,viola_real_%,aplica_dificil_%,viola_dificil_%,aplica_aleatorio_%,viola_aleatorio_%
regla,,,,,,
C1,58.5,13.3,61.3,15.6,68.2,21.0
C2,15.6,1.2,18.4,1.1,24.6,0.8
P1,4.3,3.0,4.9,3.8,6.1,5.3
P2,9.0,3.4,9.7,3.2,10.7,2.3
P3,9.0,1.6,9.7,1.9,10.7,2.4
E1,3.2,0.6,3.1,0.6,2.5,0.4
Ep1,0.0,0.0,0.0,0.0,0.0,0.0


## 3. ¿Las reglas separan lo real de lo falso?

Si las reglas capturan algo de cómo combina la gente, los outfits reales deberían tener un score más alto que los negativos. Se mide con el AUC del score (0,5 = no separa) y una prueba de Mann-Whitney.

In [5]:
filas = []
for t in ["aleatorio", "dificil"]:
    d = outfits[outfits["tipo"].isin(["real", t])]
    auc = roc_auc_score(d["real"], d["score"])
    p = mannwhitneyu(d.loc[d["real"] == 1, "score"], d.loc[d["real"] == 0, "score"], alternative="greater").pvalue
    filas.append({"negativo": t, "score_medio_real": round(d.loc[d["real"] == 1, "score"].mean(), 3),
                  "score_medio_negativo": round(d.loc[d["real"] == 0, "score"].mean(), 3),
                  "AUC_score": round(auc, 3), "p_mann_whitney": f"{p:.1e}"})
display(pd.DataFrame(filas).set_index("negativo"))
print("Outfits reales con score 7:", f"{(outfits.loc[outfits['real'] == 1, 'score'] == 7).mean():.1%}")

,score_medio_real,score_medio_negativo,AUC_score,p_mann_whitney
negativo,,,,
aleatorio,6.768,6.678,0.539,1.8e-58
dificil,6.768,6.740,0.512,2.2e-07


Outfits reales con score 7: 80.8%


## 4. Salidas

In [6]:
cols = ["outfit_id", "tipo", "real", "set_id_origen", "particion", "n_items", "cambiadas", "tids",
        *[f"aplica_{r}" for r in REGLAS], *[f"cumple_{r}" for r in REGLAS], "score"]
salida = outfits[cols].assign(tids=outfits["tids"].str.join("|"))
salida.to_csv(PROC / "fase2_outfits.csv", index=False, encoding="utf-8")
productos.to_csv(PROC / "fase2_productos.csv", index=False, encoding="utf-8")
print("Guardado: fase2_outfits.csv", salida.shape, "| fase2_productos.csv", productos.shape)
display(salida.groupby(["particion", "tipo"]).size().unstack().reindex(["train", "valid", "test"]))

Guardado: fase2_outfits.csv (45036, 23) | fase2_productos.csv (65366, 13)


tipo,aleatorio,dificil,real
particion,,,
train,10625,10625,10625
valid,2194,2194,2194
test,2193,2193,2193
